# Verifying Pandas vs SQL (PostgreSQL)

In [2]:
import pandas as pd
df = pd.read_csv('marketing_campaign_cleaned.csv')
df

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Response,Age,Total_Spend
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,...,7,0,0,0,0,0,0,1,57,1617
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,...,5,0,0,0,0,0,0,0,60,27
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,...,4,0,0,0,0,0,0,0,49,776
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,...,6,0,0,0,0,0,0,0,30,53
4,5324,1981,PhD,Married,58293.0,1,0,2014-01-19,94,173,...,5,0,0,0,0,0,0,0,33,422
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2232,10870,1967,Graduation,Married,61223.0,0,1,2013-06-13,46,709,...,5,0,0,0,0,0,0,0,47,1341
2233,4001,1946,PhD,Together,64014.0,2,1,2014-06-10,56,406,...,7,0,0,0,1,0,0,0,68,444
2234,7270,1981,Graduation,Divorced,56981.0,0,0,2014-01-25,91,908,...,6,0,1,0,0,0,0,0,33,1241
2235,8235,1956,Master,Together,69245.0,0,1,2014-01-24,8,428,...,3,0,0,0,0,0,0,0,58,843


In [3]:
print("Shape:", df.shape)
df.head()

Shape: (2237, 29)


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Response,Age,Total_Spend
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,...,7,0,0,0,0,0,0,1,57,1617
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,...,5,0,0,0,0,0,0,0,60,27
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,...,4,0,0,0,0,0,0,0,49,776
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,...,6,0,0,0,0,0,0,0,30,53
4,5324,1981,PhD,Married,58293.0,1,0,2014-01-19,94,173,...,5,0,0,0,0,0,0,0,33,422


## Verification 1: Total spend by education
**Corresponds to SQL Query 1** (JOIN + GROUP BY across the `customers` and `spending` tables).

In [4]:
# Step 1 - Verify Query 1 (total spend by education)

print(df.groupby('Education')['Total_Spend'].sum().sort_values(ascending=False))

Education
Graduation    698626
PhD           324938
Master        226359
2n Cycle      100708
Basic           4417
Name: Total_Spend, dtype: int64


## Verification 2: Latest campaign acceptance rate
**Corresponds to SQL Query 5** (aggregation on the `campaigns` table).

**Result:** matches SQL Query 1 exactly, down to the dollar, for every education group.

In [5]:
# Step 2 - Verify Query 5 (latest campaign acceptance rate)

print(round(df['Response'].mean() * 100, 2))

14.93


**Result:** 14.93 — matches SQL Query 5's `Campaign6_Latest` acceptance rate exactly.

## Verification 3: Childless high-spenders' income profile
**Corresponds to SQL Query 10** — the most structurally complex query in the set (a nested subquery). This is
the most important one to verify, since it's the query most likely to contain a subtle logic error if one
existed.

In [6]:
# Step 3 - Verify Query 10's childless high-spender count

spend_cols = ['MntWines','MntFruits','MntMeatProducts','MntFishProducts','MntSweetProducts','MntGoldProds']
df['check_total'] = df[spend_cols].sum(axis=1)
avg_total = df['check_total'].mean()
childless_high = df[(df['Kidhome']==0) & (df['Teenhome']==0) & (df['check_total'] > avg_total)]
print("Count:", len(childless_high))
print("Avg income:", round(childless_high['Income'].mean(), 2))

Count: 478
Avg income: 75955.28


**Result:** Count 478, Avg income 75,955.28 — matches SQL Query 10 exactly.

## Verification Against Pandas

| Metric | SQL Result | Pandas Result | Match |
|---|---|---|---|
| Total spend, Graduation | 698,626.00 | 698,626 | Yes |
| Total spend, Basic | 4,417.00 | 4,417 | Yes |
| Campaign6_Latest acceptance rate | 14.93% | 14.93% | Yes |
| Childless high-spenders count | 478 | 478 | Yes |
| Childless high-spenders avg income | 75,955.28 | 75,955.28 | Yes |

All three checks — spanning a simple aggregation (Query 5), a JOIN-based aggregation (Query 1), and a nested
subquery (Query 10) — matched exactly. This covers a representative range of the query complexity used across
the full set of 10, giving strong confidence that the SQL analysis as a whole is correct.